# ⚽ FutbolIA — Pipeline Profesional de Análisis Táctico con ReID

**Tracking persistente multimodal:** Kalman CA en metros reales + embeddings MobileNetV3 + histogramas HSV + Algoritmo Húngaro

---

### ▶️ Instrucciones
1. Ejecuta la **Celda 0** para ajustar las rutas a tus archivos en Google Drive.
2. Luego ejecuta las demás celdas **en orden**, de arriba hacia abajo.

---
## 🔧 CELDA 0 — CONFIGURACIÓN (¡EDITA AQUÍ ANTES DE CORRER!)
---

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║            ⚙️  CONFIGURACIÓN PRINCIPAL — EDITA AQUÍ                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── 📁 CARPETA RAÍZ del proyecto en tu Google Drive ──────────────────────
# Debe contener las carpetas: model/, clips/, src/, config/, etc.
DRIVE_PROYECTO = "/content/drive/MyDrive/Futbol2026"

# ── 🤖 MODELOS YOLO ───────────────────────────────────────────────────────
# Ruta completa en Google Drive para cada modelo
MODELO_DETECCION   = "/content/drive/MyDrive/Futbol2026/model/best.pt"
MODELO_POSE_CANCHA = "/content/drive/MyDrive/Futbol2026/model/best_pose.pt"

# ── 🎬 VIDEO A PROCESAR ───────────────────────────────────────────────────
# Ruta completa en Google Drive al video que quieres analizar
VIDEO_ENTRADA      = "/content/drive/MyDrive/Futbol2026/clips/original/clip_01.mp4"

# ── 💾 NOMBRE DEL ARCHIVO DE SALIDA ──────────────────────────────────────
# Los videos procesados se llamarán, por ejemplo: clip_01_main.mp4, clip_01_grid.mp4
# (Se toma automáticamente del nombre del video de entrada, no es necesario cambiarlo)

# ── ⚙️  PARÁMETROS DEL PIPELINE ──────────────────────────────────────────
CONFIANZA_JUGADORES = 0.25    # Umbral de confianza para detectar jugadores (0.0 - 1.0)
TAMAÑO_IMAGEN       = 640     # Tamaño de inferencia YOLO (640 recomendado, 1280 = más lento pero más preciso)

# ╔══════════════════════════════════════════════════════════════════════╗
# ║         ✅  FIN DE LA CONFIGURACIÓN — No toques lo de abajo         ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os
print("╔══════════════════════════════════════════════════════════════╗")
print("║                  CONFIGURACIÓN ACTUAL                       ║")
print("╚══════════════════════════════════════════════════════════════╝")
print(f"  📁 Proyecto     : {DRIVE_PROYECTO}")
print(f"  🤖 Detección    : {MODELO_DETECCION}")
print(f"  🏟️  Pose Cancha  : {MODELO_POSE_CANCHA}")
print(f"  🎬 Video entrada : {VIDEO_ENTRADA}")
print(f"  🎯 Confianza    : {CONFIANZA_JUGADORES}")
print(f"  🔍 Tamaño img.  : {TAMAÑO_IMAGEN}")
print()

# Verificar que los archivos existen (pre-validación antes de montar Drive)
print("═" * 62)
print("  ℹ️  Los archivos se verificarán después de montar Google Drive.")
print("  ➡️  Ejecuta la siguiente celda para continuar.")

---
## Celda 1 — Instalar dependencias

In [ ]:
!pip install -q ultralytics imageio-ffmpeg tqdm opencv-python-headless numpy scipy torch torchvision pillow
print("✅ Dependencias instaladas.")

---
## Celda 2 — Montar Google Drive y preparar archivos

In [ ]:
import shutil, os
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive', force_remount=True)

# ── Copiar código fuente desde Drive ──────────────────────────────────────
print("\n📂 Preparando código fuente...")
!rm -rf /content/FutbolIA

if os.path.exists(DRIVE_PROYECTO):
    shutil.copytree(
        DRIVE_PROYECTO,
        "/content/FutbolIA",
        ignore=shutil.ignore_patterns('.venv', '.git', '__pycache__', '*.pyc')
    )
    print(f"✅ Código copiado desde Drive: {DRIVE_PROYECTO}")
else:
    print(f"⚠️  Carpeta del proyecto no encontrada en Drive. Clonando desde GitHub...")
    !git clone https://github.com/HectrorrVas/FutbolIA.git /content/FutbolIA

# ── Crear estructura de carpetas necesaria ────────────────────────────────
for folder in ["/content/FutbolIA/model",
               "/content/FutbolIA/clips/original",
               "/content/FutbolIA/clips/processed",
               "/content/FutbolIA/output/heatmaps"]:
    os.makedirs(folder, exist_ok=True)

# ── Copiar modelos YOLO ───────────────────────────────────────────────────
print("\n🤖 Verificando modelos YOLO...")
errores = []

for src_m, nombre in [(MODELO_DETECCION, "best.pt"), (MODELO_POSE_CANCHA, "best_pose.pt")]:
    dst_m = f"/content/FutbolIA/model/{nombre}"
    if os.path.exists(src_m):
        shutil.copy2(src_m, dst_m)
        sz = os.path.getsize(dst_m) / (1024*1024)
        print(f"  ✅ {nombre:18s} → {sz:.1f} MB")
    else:
        print(f"  ❌ NO ENCONTRADO: {src_m}")
        errores.append(src_m)

# ── Copiar video de entrada ───────────────────────────────────────────────
print("\n🎬 Verificando video de entrada...")
nombre_video = os.path.basename(VIDEO_ENTRADA)
dst_video = f"/content/FutbolIA/clips/original/{nombre_video}"

if os.path.exists(VIDEO_ENTRADA):
    shutil.copy2(VIDEO_ENTRADA, dst_video)
    sz = os.path.getsize(dst_video) / (1024*1024)
    print(f"  ✅ {nombre_video:25s} → {sz:.1f} MB")
else:
    print(f"  ❌ NO ENCONTRADO: {VIDEO_ENTRADA}")
    errores.append(VIDEO_ENTRADA)

# ── Resumen ───────────────────────────────────────────────────────────────
print()
if errores:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  ❌ ERROR — Archivos no encontrados en Google Drive          ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    for e in errores:
        print(f"   → {e}")
    print("\n  ⚠️  Revisa las rutas en la CELDA 0 (CONFIGURACIÓN) y vuelve a ejecutar.")
else:
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║  ✅ Todo listo — Puedes ejecutar la Celda 3 para procesar.  ║")
    print("╚══════════════════════════════════════════════════════════════╝")

# Guardar el nombre del video para la celda de ejecución
STEM_VIDEO = os.path.splitext(nombre_video)[0]

---
## Celda 3 — Ejecutar el pipeline de análisis táctico

In [ ]:
import subprocess, sys, os

# Configurar el entorno de ejecución con las variables de la Celda 0
env = os.environ.copy()
env["PYTHONPATH"] = "/content/FutbolIA"

# Crear script de ejecución dinámico con los parámetros de la Celda 0
script = f"""
import sys
sys.path.insert(0, '/content/FutbolIA')

from pathlib import Path
from src.processor import VideoProcessor

MODELO_DETECCION   = "/content/FutbolIA/model/best.pt"
MODELO_POSE_CANCHA = "/content/FutbolIA/model/best_pose.pt"
VIDEO_ENTRADA      = "/content/FutbolIA/clips/original/{os.path.basename(VIDEO_ENTRADA)}"
VIDEO_SALIDA       = "/content/FutbolIA/clips/processed/{os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]}_main.mp4"
CONFIANZA          = {CONFIANZA_JUGADORES}
TAMAÑO_IMAGEN      = {TAMAÑO_IMAGEN}

processor = VideoProcessor(
    model_path=MODELO_DETECCION,
    pose_model_path=MODELO_POSE_CANCHA,
    confidence=CONFIANZA,
    imgsz=TAMAÑO_IMAGEN
)

processor.process(
    input_video_path=VIDEO_ENTRADA,
    output_video_path=VIDEO_SALIDA
)
"""

with open("/tmp/run_futbol.py", "w") as f:
    f.write(script)

!python /tmp/run_futbol.py

---
## Celda 4 — Visualizar los videos procesados

In [ ]:
import os
from IPython.display import HTML, display
from base64 import b64encode

def show_video(path, title=""):
    if not os.path.exists(path):
        print(f"⚠️  No encontrado: {path}")
        return
    sz = os.path.getsize(path) / (1024*1024)
    data = open(path, 'rb').read()
    url  = "data:video/mp4;base64," + b64encode(data).decode()
    display(HTML(f"""
        <h3 style='color:#ddd;font-family:sans-serif'>{title} &nbsp;
        <small style='color:#888'>({sz:.1f} MB)</small></h3>
        <video width='900' controls style='border-radius:8px;margin-bottom:20px'>
            <source src='{url}' type='video/mp4'>
        </video>
    """))

stem = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]
base = "/content/FutbolIA/clips/processed"

show_video(f"{base}/{stem}_grid.mp4",    "⚽ Grid 2x2 — Todas las vistas")
show_video(f"{base}/{stem}_main.mp4",    "📊 Vista General + Mapa 2D Métrico (ReID)")
show_video(f"{base}/{stem}_equipoA.mp4", "🔵 Análisis Táctico — Equipo A")
show_video(f"{base}/{stem}_equipoB.mp4", "🔴 Análisis Táctico — Equipo B")

---
## Celda 5 — Descargar los videos procesados a tu PC

In [ ]:
import os
from google.colab import files

stem = os.path.splitext(os.path.basename(VIDEO_ENTRADA))[0]
base = "/content/FutbolIA/clips/processed"

videos = [
    (f"{base}/{stem}_grid.mp4",    "Grid 2x2"),
    (f"{base}/{stem}_main.mp4",    "Main"),
    (f"{base}/{stem}_equipoA.mp4", "Equipo A"),
    (f"{base}/{stem}_equipoB.mp4", "Equipo B"),
]

for path, label in videos:
    if os.path.exists(path):
        print(f"⬇️  Descargando: {label}...")
        files.download(path)
    else:
        print(f"⚠️  No encontrado: {label}")